In [ ]:
# Dataset sanity checks after load_features

from IPython.display import display

print("Shape:", df.shape)
print("Unique contact_id:", df["contact_id"].nunique())
print("Duplicated contact_id:", df["contact_id"].duplicated().sum())

assert df["contact_id"].is_unique, "Есть дубли contact_id после merge фичей"

# Target checks
target_col = "target_churn_from_dac"

if target_col in df.columns:
    print("\nTarget distribution:")
    display(df[target_col].value_counts(dropna=False).to_frame("cnt"))
    display(df[target_col].value_counts(normalize=True, dropna=False).to_frame("share"))

    assert df[target_col].isna().sum() == 0, "В target есть NaN"
    assert set(df[target_col].dropna().unique()).issubset({0, 1}), "Target должен быть бинарным 0/1"

# Required features checks
missing_features = sorted(set(features) - set(df.columns))
extra_features = sorted(set(df.columns) - set(features) - {"contact_id", target_col, "is_dac_next_month"})

print("\nMissing features:", len(missing_features))
display(missing_features[:50])

assert len(missing_features) == 0, f"Не хватает фичей: {missing_features}"

# NaN checks
na_report = (
    df[features]
    .isna()
    .mean()
    .sort_values(ascending=False)
    .to_frame("na_share")
)

print("\nTop missing features:")
display(na_report.head(30))

# Fully empty features
empty_features = na_report[na_report["na_share"] == 1].index.tolist()
print("\nFully empty features:", len(empty_features))
display(empty_features)

assert len(empty_features) == 0, f"Есть полностью пустые фичи: {empty_features}"

# Constant features
constant_features = [
    col for col in features
    if df[col].nunique(dropna=False) <= 1
]

print("\nConstant features:", len(constant_features))
display(constant_features)

# Numeric finite checks
numeric_features = df[features].select_dtypes(include=["number"]).columns.tolist()

inf_report = (
    np.isinf(df[numeric_features])
    .sum()
    .sort_values(ascending=False)
    .to_frame("inf_cnt")
)

inf_report = inf_report[inf_report["inf_cnt"] > 0]

print("\nFeatures with inf:")
display(inf_report)

assert len(inf_report) == 0, "Есть inf/-inf в числовых фичах"

# Duplicate columns check
duplicated_columns = df.columns[df.columns.duplicated()].tolist()
print("\nDuplicated columns:", duplicated_columns)

assert len(duplicated_columns) == 0, f"Есть дубли колонок: {duplicated_columns}"

# Basic descriptive stats for important blocks
key_cols = [
    "cheque_recency",
    "login_recency",
    "omni_qr_recency",
    "omni_features_recency",
    "perf_recency",
    "dac_months_count",
    "dac_months_last_3",
    "dac_months_last_6",
    "dac_months_last_12",
    "dac_share_last_12",
    "transaction_tendency",
    "login_tendency",
]

key_cols = [c for c in key_cols if c in df.columns]

print("\nKey features describe:")
display(df[key_cols].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).T)

print("\nOK: dataset sanity checks passed")